# 00. Verify the data sources (PetrIA-Ingest)

**Purpose.** The Day 2 research report is a list of *claims* about the official
datasets. It could not open the portal, so every claim is unverified. This
notebook checks each claim against the real files and prints one report.

**How to run (Google Colab).** Runtime → Run all. It takes a few minutes and
downloads at most a few hundred MB into the Colab disk.

**What it does not do.** It writes nothing to any database and changes nothing
outside Colab. It never invents a value: if a check fails, the failure goes into
the report under `errors`.

**What to send back.** The last cell prints the full report. Copy the whole
output and paste it in the chat.

In [ ]:
%pip -q install pyogrio pyproj requests pandas

In [ ]:
import codecs
import hashlib
import json
import re
import zipfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

WORKDIR = Path("data_verification")
WORKDIR.mkdir(exist_ok=True)
HEADERS = {"User-Agent": "petria-verification-notebook/1.0"}
MAX_DOWNLOAD_MB = 400

# Portals to try, in order. The research says both run CKAN (UNVERIFIED).
HOSTS = ["https://datos.gob.ar", "https://datos.energia.gob.ar"]

# Candidate dataset slugs copied from the research report (UNVERIFIED).
# This notebook checks each one against the live portal.
CANDIDATE_SLUGS = {
    "production_chapter_iv": [
        "energia-produccion-petroleo-gas-por-pozo-capitulo-iv",
        "produccion-de-petroleo-y-gas-por-pozo",
    ],
    "fracturing_adjunto_iv": [
        "energia-datos-fractura-pozos-hidrocarburos-adjunto-iv",
        "datos-de-fractura-de-pozos-adjunto-iv",
    ],
    "concessions": [
        "energia-produccion-hidrocarburos---concesiones-explotacion",
        "energia-produccion-hidrocarburos---lotes-explotacion",
        "energia-exploracion-hidrocarburos---permisos-exploracion",
    ],
}
SEARCH_QUERIES = [
    "capitulo iv pozos",
    "fractura adjunto iv",
    "concesiones explotacion",
    "lotes explotacion",
    "permisos exploracion",
]

REPORT = {
    "started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "errors": [],
    "datasets": {},
    "search": {},
    "downloads": {},
    "profiles": {},
    "joins": {},
    "claims_check": {},
}

## 1. Ask the portal what it really has
CKAN is the software behind the portal. Its API tells us, for each dataset, the
license, the last modification and the list of files. `ckan()` tries each host
and records every failure instead of stopping.

In [ ]:
def log_error(where, exc):
    REPORT["errors"].append({"where": where, "error": f"{type(exc).__name__}: {exc}"[:300]})


def ckan(action, **params):
    """Call a CKAN action on each host until one answers: (host, result) or (None, None)."""
    for host in HOSTS:
        try:
            response = requests.get(
                f"{host}/api/3/action/{action}", params=params, headers=HEADERS, timeout=30
            )
            response.raise_for_status()
            body = response.json()
            if body.get("success"):
                return host, body["result"]
            log_error(f"{action} {params} @ {host}", RuntimeError("success is false"))
        except Exception as exc:
            log_error(f"{action} {params} @ {host}", exc)
    return None, None


def summarize_package(pkg):
    return {
        "title": pkg.get("title"),
        "license_id": pkg.get("license_id"),
        "license_title": pkg.get("license_title"),
        "metadata_modified": pkg.get("metadata_modified"),
        "extras": {e.get("key"): e.get("value") for e in pkg.get("extras", [])},
        "resources": [
            {
                "id": r.get("id"),
                "name": r.get("name"),
                "format": r.get("format"),
                "url": r.get("url"),
                "size": r.get("size"),
                "last_modified": r.get("last_modified") or r.get("metadata_modified"),
            }
            for r in pkg.get("resources", [])
        ],
    }

In [ ]:
for label, slugs in CANDIDATE_SLUGS.items():
    REPORT["datasets"][label] = {}
    for slug in slugs:
        host, pkg = ckan("package_show", id=slug)
        REPORT["datasets"][label][slug] = (
            {"host": host, **summarize_package(pkg)} if pkg else {"found": False}
        )

for query in SEARCH_QUERIES:
    host, result = ckan("package_search", q=query, rows=8)
    REPORT["search"][query] = (
        [
            {
                "name": p.get("name"),
                "title": p.get("title"),
                "license_title": p.get("license_title"),
                "metadata_modified": p.get("metadata_modified"),
            }
            for p in result.get("results", [])
        ]
        if result
        else []
    )

rows = [
    {
        "dataset": label,
        "slug": slug,
        "found": info.get("found", True),
        "license": info.get("license_title"),
        "modified": info.get("metadata_modified"),
        "n_resources": len(info.get("resources", [])),
    }
    for label, by_slug in REPORT["datasets"].items()
    for slug, info in by_slug.items()
]
pd.DataFrame(rows)

## 2. Choose which files to inspect
We pick resources by name, per dataset. **Check the table this cell prints.** If
a pick is wrong or missing, set `OVERRIDE_URLS` (for example
`{"registry_csv": "https://..."}`) and run again.

In [ ]:
resources = [
    {"dataset": label, "slug": slug, **r}
    for label, by_slug in REPORT["datasets"].items()
    for slug, info in by_slug.items()
    for r in info.get("resources", [])
]

# key: (dataset label, regex on the resource name, accepted formats)
PICK = {
    "registry_csv": ("production_chapter_iv", r"cap[ií]tulo iv\s*-\s*pozos", {"csv"}),
    "wells_listing_csv": ("production_chapter_iv", r"listado de pozos", {"csv"}),
    "production_unconventional_csv": ("production_chapter_iv", r"no convencional", {"csv"}),
    "wells_shapefile": ("production_chapter_iv", r"shapefile", {"zip", "shp"}),
    "fracturing_csv": ("fracturing_adjunto_iv", r"fractura|adjunto", {"csv"}),
    "concessions_shp": ("concessions", r"concesi", {"zip", "shp"}),
    "lots_shp": ("concessions", r"lotes", {"zip", "shp"}),
    "permits_shp": ("concessions", r"permisos", {"zip", "shp"}),
}
OVERRIDE_URLS = {}


def pick_resources():
    picked = {}
    for key, (label, pattern, formats) in PICK.items():
        for r in resources:
            haystack = f"{r.get('name') or ''} {r['slug']}"
            fmt = (r.get("format") or "").lower()
            if r["dataset"] == label and re.search(pattern, haystack, re.I) and fmt in formats:
                picked[key] = r
                break
    for key, url in OVERRIDE_URLS.items():
        picked[key] = {"name": "override", "format": Path(url).suffix.lstrip("."), "url": url, "size": None}
    return picked


picked = pick_resources()
print("NOT picked:", sorted(set(PICK) - set(picked)) or "none")
pd.DataFrame(
    [{"key": k, "resource": r.get("name"), "format": r.get("format"), "size": r.get("size"), "url": r.get("url")} for k, r in picked.items()]
)

## 3. The tools: download, profile a CSV, inspect a shapefile
Three small functions. `profile_csv` reads the file in chunks, so memory stays
low. It counts rows and nulls, lists the real distinct values of the formation
and resource columns, checks the coordinates, and looks for duplicate
(`idpozo`, `anio`, `mes`) keys. `inspect_geo` reads the real coordinate system
from the `.prj` file and the real attribute names.

In [ ]:
def download(url, dest, max_mb=MAX_DOWNLOAD_MB):
    """Stream a file to disk, hash it, and stop if it is too large."""
    sha, size = hashlib.sha256(), 0
    with requests.get(url, headers=HEADERS, stream=True, timeout=60, allow_redirects=True) as response:
        response.raise_for_status()
        with open(dest, "wb") as handle:
            for chunk in response.iter_content(1 << 20):
                size += len(chunk)
                if size > max_mb * (1 << 20):
                    raise RuntimeError(f"file is larger than {max_mb} MB; stopped")
                sha.update(chunk)
                handle.write(chunk)
        return {
            "path": str(dest),
            "bytes": size,
            "sha256": sha.hexdigest(),
            "content_type": response.headers.get("content-type"),
            "final_url": response.url,
        }


CATEGORICAL = {"formacion", "formprod", "tipo_de_recurso", "sub_tipo_recurso", "cuenca",
               "provincia", "tipoestado", "tipopozo", "clasificacion", "subclasificacion"}
COORDS = ("coordenadax", "coordenaday")


def sniff(path):
    """Guess the encoding and the delimiter from the first bytes."""
    raw = Path(path).read_bytes()[:4096]
    text, encoding = "", "latin-1"
    for candidate in ("utf-8-sig", "latin-1"):
        try:
            text = codecs.getincrementaldecoder(candidate)().decode(raw, final=False)
            encoding = candidate
            break
        except UnicodeDecodeError:
            continue
    first = text.splitlines()[0] if text else ""
    return encoding, max([",", ";", "\t", "|"], key=first.count)


def profile_csv(path, chunksize=200_000, top=15):
    """Return (profile, key_sets). Everything is read as text: no silent type guessing."""
    encoding, sep = sniff(path)
    header = pd.read_csv(path, sep=sep, encoding=encoding, nrows=0).columns.tolist()
    lower = {c.lower(): c for c in header}
    n_rows, nulls = 0, Counter()
    distinct = {lower[c]: Counter() for c in CATEGORICAL if c in lower}
    coord = {lower[c]: {"min": None, "max": None, "null": 0, "zero": 0} for c in COORDS if c in lower}
    keys = {name: set() for name in ("idpozo", "sigla") if name in lower}
    composite = [lower[c] for c in ("idpozo", "anio", "mes") if c in lower]
    seen, duplicate_rows = set(), 0

    for chunk in pd.read_csv(path, sep=sep, encoding=encoding, dtype=str, chunksize=chunksize, low_memory=False):
        n_rows += len(chunk)
        nulls.update({k: int(v) for k, v in chunk.isna().sum().items()})
        for col, counter in distinct.items():
            counter.update(chunk[col].fillna("<NA>").value_counts().to_dict())
        for col, stats in coord.items():
            values = pd.to_numeric(chunk[col].str.replace(",", ".", regex=False), errors="coerce")
            stats["null"] += int(values.isna().sum())
            stats["zero"] += int((values == 0).sum())
            if values.notna().any():
                low, high = float(values.min()), float(values.max())
                stats["min"] = low if stats["min"] is None else min(stats["min"], low)
                stats["max"] = high if stats["max"] is None else max(stats["max"], high)
        for name, store in keys.items():
            store.update(chunk[lower[name]].dropna().unique())
        if len(composite) == 3:
            for row in zip(*(chunk[c] for c in composite)):
                if row in seen:
                    duplicate_rows += 1
                else:
                    seen.add(row)

    profile = {
        "encoding": encoding,
        "separator": sep,
        "n_rows": n_rows,
        "n_columns": len(header),
        "columns": header,
        "null_counts": {k: v for k, v in nulls.items() if v},
        "distinct_values": {
            col: {"n_distinct": len(c), "top": dict(c.most_common(top))} for col, c in distinct.items()
        },
        "coordinate_stats": coord,
        "n_unique_keys": {name: len(store) for name, store in keys.items()},
        "duplicate_idpozo_anio_mes_rows": duplicate_rows if len(composite) == 3 else "not computed",
        "sample_rows": pd.read_csv(path, sep=sep, encoding=encoding, dtype=str, nrows=3).to_dict("records"),
    }
    return profile, keys


def inspect_geo(path):
    """Read the real CRS, geometry type, feature count and attribute names of a zipped shapefile."""
    from pyogrio import list_layers, read_dataframe, read_info
    from pyproj import CRS

    out = {"prj_files": {}, "layers": {}}
    with zipfile.ZipFile(path) as zf:
        for name in zf.namelist():
            if name.lower().endswith(".prj"):
                text = zf.read(name).decode("utf-8", "replace")
                try:
                    epsg = CRS.from_wkt(text).to_epsg()
                except Exception:
                    epsg = None
                out["prj_files"][name] = {"epsg_guess": epsg, "wkt_start": text[:160]}
    for layer_name, _ in list_layers(path):
        info = read_info(path, layer=layer_name)
        sample = read_dataframe(path, layer=layer_name, max_features=3)
        bounds = info.get("total_bounds")
        out["layers"][layer_name] = {
            "geometry_type": info.get("geometry_type"),
            "features": int(info["features"]),
            "crs": info.get("crs"),
            "fields": list(info["fields"]),
            "bounds": [float(x) for x in bounds] if bounds is not None else None,
            "sample": sample.drop(columns="geometry").astype(str).to_dict("records"),
        }
    return out

## 4. Download and inspect the picked files

In [ ]:
KEY_SETS = {}
for key, r in picked.items():
    url = r.get("url")
    suffix = ".zip" if key.endswith("_shp") or key.endswith("_shapefile") else ".csv"
    dest = WORKDIR / f"{key}{suffix}"
    try:
        REPORT["downloads"][key] = download(url, dest)
        if suffix == ".csv":
            REPORT["profiles"][key], KEY_SETS[key] = profile_csv(dest)
        else:
            REPORT["profiles"][key] = inspect_geo(dest)
    except Exception as exc:
        log_error(f"download or profile: {key}", exc)

print("profiled:", sorted(REPORT["profiles"]))
print("errors so far:", len(REPORT["errors"]))

## 5. Do the datasets join? And do the research claims hold?
Two checks. **Joins:** how many wells of one file appear in another, and how many
rows per well. **Claims:** the columns the research expected versus the real header.

In [ ]:
def overlap(a, b):
    both = len(a & b)
    return {
        "n_a": len(a),
        "n_b": len(b),
        "in_both": both,
        "pct_a_found_in_b": round(100 * both / len(a), 2) if a else None,
        "pct_b_found_in_a": round(100 * both / len(b), 2) if b else None,
    }


PAIRS = [
    ("fracturing_csv", "registry_csv"),
    ("fracturing_csv", "production_unconventional_csv"),
    ("production_unconventional_csv", "registry_csv"),
]
for a, b in PAIRS:
    name = f"{a}.idpozo vs {b}.idpozo"
    if "idpozo" in KEY_SETS.get(a, {}) and "idpozo" in KEY_SETS.get(b, {}):
        REPORT["joins"][name] = overlap(KEY_SETS[a]["idpozo"], KEY_SETS[b]["idpozo"])
    else:
        REPORT["joins"][name] = "not computed: file or column missing"

registry = REPORT["profiles"].get("registry_csv")
if registry:
    REPORT["joins"]["registry rows vs unique idpozo vs unique sigla"] = {
        "rows": registry["n_rows"],
        "unique_idpozo": registry["n_unique_keys"].get("idpozo"),
        "unique_sigla": registry["n_unique_keys"].get("sigla"),
    }
frac = REPORT["profiles"].get("fracturing_csv")
if frac:
    REPORT["joins"]["fracturing rows vs unique idpozo"] = {
        "rows": frac["n_rows"],
        "unique_idpozo": frac["n_unique_keys"].get("idpozo"),
    }

# Claims copied from the research report (UNVERIFIED until compared here).
CLAIMS = {
    "registry_csv": ["idpozo", "sigla", "formprod", "coordenadax", "coordenaday"],
    "fracturing_csv": [
        "idpozo", "longitud_rama_horizontal_m", "cantidad_fracturas",
        "arena_bombeada_nacional_tn", "arena_bombeada_importada_tn",
        "co2_inyectado_m3", "duracion_fractura_dias",
    ],
    "production_unconventional_csv": [
        "idpozo", "sigla", "anio", "mes", "prod_pet", "prod_gas", "prod_agua", "tef",
        "formprod", "formacion", "cuenca", "provincia", "tipo_de_recurso",
        "sub_tipo_recurso", "rectificado", "habilitado", "coordenadax", "coordenaday",
    ],
}
for key, expected in CLAIMS.items():
    profile = REPORT["profiles"].get(key)
    if not profile:
        REPORT["claims_check"][key] = "no profile (file not downloaded)"
        continue
    actual = [c.lower() for c in profile["columns"]]
    REPORT["claims_check"][key] = {
        "n_columns": len(actual),
        "expected_but_missing": sorted(set(expected) - set(actual)),
        "present_but_not_in_claims": sorted(set(actual) - set(expected)),
    }
REPORT["claims_check"]

## 6. The report to paste back
Copy the whole output of this cell into the chat. It is also saved as
`verification_report.json` in the Colab files panel.

In [ ]:
REPORT["finished_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
text = json.dumps(REPORT, indent=1, ensure_ascii=False, default=str)
Path("verification_report.json").write_text(text, encoding="utf-8")
print(text)